In [3]:
!wget https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt

--2026-03-06 17:29:36--  https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... failed: Connection timed out.
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... Read error (Connection timed out) in headers.
Retrying.

--2026-03-06 17:30:13--  (try: 2)  https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M   372KB/s    in 2.9s    

2026-03-06 17:30:17 (372 KB/s

In [230]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [228]:
with open('input_cn.txt', 'r') as f:
    text=f.read()

In [229]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1942


In [3]:
print(text[:100])

人工智能的诞生与发展

人工智能（Artificial Intelligence，简称AI）是一门旨在使机器能够模拟人类智能行为的技术科学。它的起源可以追溯到20世纪中叶，1956年的达特茅斯会议被公


In [4]:
chars = sorted(list(set(text)))
print(''.join(chars))
vocab_size = len(chars)
print(vocab_size)


 -01256789ABCEFGINOPRSTYacdefghilmnoprstuvwxy·“”、。《》一上下不与专世业个中临为主义也习了于些人今仍从他代令以们件任伐优会传伦但位低何作使依保信借偏像允元先入全公共关兴其具内再写军冠冬决准凭出分列则创初利别到制前力务动助勇包化医升单卡卷原参双反发取受变只可叶号各合同后向命和响喜器回因团围图在地场型域基堂处备复外多够大夺奇奠好如始学它安完定实家容寒对导将尝层展工巨己已布带幅平年并序应度建开异弃式强归当很律得循微心必志念思性息情意成我或战所手才扎批技拟括持指挑捉捕据掌探接推提握摒撑播支改效数文断斯新方无日旨时明是景智更最月有望朝期未本术机杂李来构果架标核根框梯棋概模次歌正此步段每比求法注活深渐源溯潜点热焦然版特献率环现球理生用由界疗疾病的益监目相真着矩石研破础硬确示社神离秀私种科积称程究突符第等答简算管类精系素索约级纪线练经结络统继续综编网置翰考者而聚背胜能脑自致色茅荐虽行表被要见规视觉解言计认让训议许论证识诊试诞语说诸调谷贡质赖赛走起足跟路践辅辑达过迎运还这进连追送适逐通速逻遇都采释重量锡键长门问队阵阶际降限险陷随隐难需面革音顺预领题风飞首马驶驾验高麦齐（），；
512


In [5]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

sample_sentence = "hii there"
print(encode(sample_sentence))
print(decode(encode(sample_sentence)))

[31, 32, 32, 1, 40, 31, 28, 38, 28]
hii there


In [6]:

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data)


torch.Size([1942]) torch.int64
tensor([ 72, 205, 283,  ..., 131, 473,  50])


In [231]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([ 72, 205, 283, 406, 345, 435, 338,  57, 147])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is {target}")

when input is tensor([72]) the target is 205
when input is tensor([ 72, 205]) the target is 283
when input is tensor([ 72, 205, 283]) the target is 406
when input is tensor([ 72, 205, 283, 406]) the target is 345
when input is tensor([ 72, 205, 283, 406, 345]) the target is 435
when input is tensor([ 72, 205, 283, 406, 345, 435]) the target is 338
when input is tensor([ 72, 205, 283, 406, 345, 435, 338]) the target is 57
when input is tensor([ 72, 205, 283, 406, 345, 435, 338,  57]) the target is 147


In [10]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(f"ix.shape:\n{ix.shape}\n{ix}\n{[data[i:i+block_size] for i in ix]}\n{torch.stack([data[i:i+block_size] for i in ix])}\n=====")
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(f'inputs:\n{xb.shape}\n{xb}')
print(f'targets:\n{yb.shape}\n{yb}')
print('=====')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[203, 361, 389, 397, 391, 390, 298, 161],
        [ 38, 300, 298, 510,  53, 381, 122,  84],
        [  5, 213, 510,  11,  33,  28,  44,  18],
        [204, 335, 120,  69,  53, 194, 345, 283]])
targets:
torch.Size([4, 8])
tensor([[361, 389, 397, 391, 390, 298, 161, 146],
        [300, 298, 510,  53, 381, 122,  84, 363],
        [213, 510,  11,  33,  28,  44,  18,  28],
        [335, 120,  69,  53, 194, 345, 283, 406]])
=====
when input is [203] the target: 361
when input is [203, 361] the target: 389
when input is [203, 361, 389] the target: 397
when input is [203, 361, 389, 397] the target: 391
when input is [203, 361, 389, 397, 391] the target: 390
when input is [203, 361, 389, 397, 391, 390] the target: 298
when input is [203, 361, 389, 397, 391, 390, 298] the target: 161
when input is [203, 361, 389, 397, 391, 390, 298, 161] the target: 146
when input is [38] the target: 300
when input is [38, 300] the target: 298
when input is [38, 300, 298] the t

In [11]:
print(xb) # our input(embedding) to the transformer

tensor([[203, 361, 389, 397, 391, 390, 298, 161],
        [ 38, 300, 298, 510,  53, 381, 122,  84],
        [  5, 213, 510,  11,  33,  28,  44,  18],
        [204, 335, 120,  69,  53, 194, 345, 283]])


In [12]:
torch.manual_seed(1377)


class BigramLanguageModel(nn.Module):
    '''
    Bigram（二元语法）是最简单的语言模型，也是理解 Transformer/GPT 的 “入门钥匙”
    只依赖前一个 Token 预测下一个 Token
    通常只有一个嵌入层（embedding table）,维度为 [vocab_size, vocab_size]
    
    训练目标：让模型学会 “常见的相邻 Token 组合”（比如 “人工→智能”“机器→学习”），
    但无法捕捉长程依赖（比如 “我昨天去公园，看到了一只…→猫” 这种跨多个 Token 的关联）
    '''
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # print(f"B = {B}, T = {T}, C = {C}") # 4, 8, 65
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1)
        return idx   


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(f"logits.shape: {logits.shape}, loss: {loss}") # -ln(1/512), 约等于 6.238

print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))



logits.shape: torch.Size([32, 512]), loss: 7.051459789276123

谷齐题s与面提目置综带降背诸线T版率》荐共务队神望合目谷围践隐冬n性够限型76得信t才助围聚命得r业翰者觉新或着说飞版当
赛·性结程证信源核要数矩断参和翰赖段随仍胜还写此络内带杂先再规，通献世言i石文


In [13]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [15]:
batch_size = 32
for steps in range(1000):
    xb, yb = get_batch('train')

    logtis, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

4.429830074310303


In [19]:
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


驶门开反风考传只荐让制建路智图带智。
律下工并辅语认R背围路入医指家研再医预的都诸表此达诞业致准而G今结堂将单依称效务一概创追背概素成创C此力日秀迎赖实解t算寒一石家意理持版勇念列学，难诊拟齐w奠B遇


## 自注意力的一些数学技巧

In [213]:
torch.manual_seed(1337)
'''
B = 4：Batch Size（一次处理 4 个样本）
T = 8：Sequence Length（每个样本有 8 个 Token，时间步）
C = 2：Channel/Embedding Size（每个 Token 用 2 维向量表示）
'''
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

### 版本 1

In [214]:
xbow = torch.zeros((B, T, C))
for b in range(B): # 遍历每个样本（批量维度）
    for t in range(T): # 遍历每个时间步（序列维度）
        xprev = x[b, :t+1] # 取当前样本中，0~t 共 t+1 个 Token → 形状 [t+1, C]。这一步模拟了：在第 t 步，只能看到前面所有 Token（包括自己），看不到后面的 Token
        xbow[b, t] = torch.mean(xprev, 0) # 对这 t+1 个 Token 做「逐维度平均」。这一步模拟了：用前面所有 Token 的 “平均特征” 来表示当前第 t 个 Token 的上下文

# x[0]
# tensor([[ 0.1808, -0.0700],
#         [-0.3596, -0.9152],
#         [ 0.6258,  0.0255],
#         [ 0.9545,  0.0643],
#         [ 0.3612,  1.1679],
#         [-1.3499, -0.5102],
#         [ 0.2360, -0.2398],
#         [-0.9211,  1.5433]])

# xbow[0]
# tensor([[ 0.1808, -0.0700],
#         [-0.0894, -0.4926],
#         [ 0.1490, -0.3199],
#         [ 0.3504, -0.2238], # 解释示例: [..., (-0.0700-0.9152+0.0255+0.0643)/4]
#         [ 0.3525,  0.0545],
#         [ 0.0688, -0.0396],
#         [ 0.0927, -0.0682],
#         [-0.0341,  0.1332]])

### 版本 2

In [215]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)

#wei
# tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
#         [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
#         [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
#         [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

xbow2 = wei @ x # (B, T, T) @ (B, T, C) -> (B, T, C)
torch.allclose(xbow, xbow2) # False
diff = torch.abs(xbow - xbow2)
print("最大差值：", diff.max())
print("差值超过阈值的元素：", (diff > 1e-05).nonzero())
# 最大差值： tensor(3.2363e-08)
# 差值超过阈值的元素： tensor([], size=(0, 3), dtype=torch.int64)

torch.allclose(xbow, xbow2, rtol=1e-6, atol=1e-7) # 和版本 1 基本相同

最大差值： tensor(3.2363e-08)
差值超过阈值的元素： tensor([], size=(0, 3), dtype=torch.int64)


True

### 版本 3 with softmax

In [216]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
diff = torch.abs(xbow - xbow3)
print(torch.allclose(xbow, xbow3, rtol=1e-6, atol=1e-7))

True


In [217]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [223]:
wei = torch.zeros((T, T))
wei

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [224]:
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [222]:
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4955, 0.5045, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3267, 0.3326, 0.3406, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2416, 0.2460, 0.2519, 0.2605, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1897, 0.1931, 0.1978, 0.2045, 0.2150, 0.0000, 0.0000, 0.0000],
        [0.1538, 0.1565, 0.1603, 0.1657, 0.1742, 0.1894, 0.0000, 0.0000],
        [0.1257, 0.1279, 0.1310, 0.1354, 0.1424, 0.1548, 0.1828, 0.0000],
        [0.0966, 0.0983, 0.1007, 0.1041, 0.1094, 0.1189, 0.1405, 0.2316]])

### 版本4 自注意力

In [250]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# 看看单头实现的自注意力
head_size = 16
key = nn.Linear(C, head_size, bias=False)

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

key



Linear(in_features=32, out_features=16, bias=False)